# The command-line tools

wfdb-x includes all the command-line tools of the WFDB package (`rdsamp`,
`rdann`, `wfdbdesc`, `ann2rr`, `xform`, `snip`, `mit2edf`, ...), with the
same options and output. In a shell, run them with the `wfdbx` command:

```
wfdbx rdsamp -r 100s -t 0:5 -v -p
wfdbx --list
```

From Python (here, in a notebook), run them with `subprocess`. The helper
below runs a tool with the sample data on the WFDB path.

In [ ]:
import os
import subprocess
import sys

import wfdbx

ENV = dict(os.environ, WFDB=". " + wfdbx.sample_data_dir())


def tool(*args, stdin=None):
    """Run a wfdb-x tool and print its output."""
    result = subprocess.run([sys.executable, "-m", "wfdbx.cli", *args],
                            input=stdin, capture_output=True, text=True,
                            env=ENV)
    print(result.stdout + result.stderr, end="")
    return result.returncode

In [ ]:
tool("wfdbdesc", "100s")

`rdsamp` prints samples; `-p` prints them in physical units, `-v` adds
column headings. (Units the header does not give are shown as `nd`, "not
defined".)

In [ ]:
tool("rdsamp", "-r", "100s", "-t", "s5", "-v", "-p")

In [ ]:
tool("rdann", "-r", "100s", "-a", "atr", "-t", "0:3")

`ann2rr` prints RR intervals; `-i s` prints them in seconds:

In [ ]:
tool("ann2rr", "-r", "100s", "-a", "atr", "-i", "s", "-t", "0:5")

## Writing a record from text

`wrsamp` writes a record from columns of numbers. The sampling frequency
(`-F`), gains (`-G`), ADC resolution (`-R`) and units (`-U`) must be given:

In [ ]:
text = "".join(f"{i} {i % 7 - 3}\n" for i in range(20))
tool("wrsamp", "-o", "fromtext", "-F", "100", "-G", "200", "-R", "16",
     "-U", "mV", stdin=text)
tool("wfdbdesc", "fromtext")

In [ ]:
# Without -F, wrsamp stops with an error (status 1):
tool("wrsamp", "-o", "fromtext", stdin=text)